In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Load dataset (grayscale, resize)

In [2]:
UKURAN = 250
data_path = "Rice"
kelas = sorted(os.listdir(data_path))
print(kelas)

x, y = [], []
for label, nama_kelas in enumerate(kelas):
    folder = os.path.join(data_path, nama_kelas)
    print(folder)

    for nama_file in os.listdir(folder):
        if nama_file.endswith("Thumbs.db"):
            continue
        gambar = keras.utils.load_img(os.path.join(folder, nama_file), color_mode="grayscale", target_size=(UKURAN, UKURAN, 1))
        x.append(np.array(gambar))
        y.append(label)

x = np.array(x)
y = np.array(y)

['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']
Rice\Arborio
Rice\Basmati
Rice\Ipsala
Rice\Jasmine
Rice\Karacadag


In [3]:
print("Jumlah gambar: ", len(x))
print("Ukuran x:", x.shape)
print("Ukuran y:", y.shape)

Jumlah gambar:  500
Ukuran x: (500, 250, 250)
Ukuran y: (500,)


# 2. Preprocessing (normalisasi + one hot)

In [4]:
x = x.astype("float32") / 255.0
y = keras.utils.to_categorical(y, num_classes=len(kelas))

print(x.shape)
print(y.shape)
print(y[0])

(500, 250, 250)
(500, 5)
[1. 0. 0. 0. 0.]


# 3. data split 80 10 10

In [6]:
# 80 train 20 temp
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# 20 temp -> 10 val, 10 test
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(len(x_train))
print(len(x_test))
print(len(x_val))

400
50
50


# 4. Model DCNN

In [7]:
model = keras.Sequential([
    # input
    keras.layers.Input(shape=(UKURAN, UKURAN, 1)),

    # hidden
    keras.layers.Conv2D(32, (3,3), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2,2), strides=2),

    keras.layers.Conv2D(64, (3,3), activation="relu"),
    keras.layers.Conv2D(64, (3,3), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2,2), strides=2),

    keras.layers.Flatten(),
    
    # output
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(len(kelas), activation="softmax"),
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 248, 248, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 124, 124, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 122, 122, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 120, 120, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 60, 60, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 230400)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    29,491,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,555,653 (112.75 MB)

 Trainable params: 29,555,653 (112.75 MB)

 Non-trainable params: 0 (0.00 B)

# 5. Training

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=32, epochs=10, validation_data=(x_val, y_val))

# 6. Show Prediction (9 gambar test)

In [ ]:
prediksi = model.predict(x_test)
tebakan = np.argmax(prediksi, axis=1)
real = np.argmax(y_test, axis=1)

plt.figure(figsize=(15, 15))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_test[i], cmap="gray")
    plt.title(f"Tebakan: {kelas[tebakan[i]]} \n Asli: {kelas[real[i]]}")
    plt.axis("off")
plt.show()

In [ ]:
for i in range(len(x_test)):
    print(f"Tebakan: {kelas[tebakan[i]]} | Asli: {kelas[real[i]]}")

# 7. evaluate

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print("Akurasi di data test:", accuracy)
print("Loss di data test:", loss)

# 8. plotting accuracy and loss

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="validation accuracy")
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.title("Loss")
plt.legend()

plt.show()